# DeepSeek 在线模型调用

这个 Notebook 对比底层 OpenAI 兼容 SDK、推荐的 `ChatDeepSeek` 和兼容写法 `ChatOpenAI`。执行包含 `invoke` 的单元会访问真实 DeepSeek API，并可能产生费用。

In [4]:
import os

from langchain_demo.config import load_project_environment, require_environment_variable

load_project_environment(override=True)
api_key = require_environment_variable("DEEPSEEK_API_KEY")
base_url = os.getenv("DEEPSEEK_API_BASE", "https://api.deepseek.com")

## 1. 使用 OpenAI SDK 直接调用 DeepSeek

这种方式最接近 HTTP API，适合理解 LangChain 模型适配器下层发生了什么。

In [5]:
from openai import OpenAI

client = OpenAI(api_key=api_key, base_url=base_url)
response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {"role": "system", "content": "You are a helpful translator."},
        {"role": "user", "content": "把“你好”翻译成日语。"},
    ],
)
print(response.choices[0].message.content)

こんにちは。


## 2. 使用 ChatDeepSeek（推荐）

`ChatDeepSeek` 能保留 DeepSeek 特有响应，并提供 LangChain 的 invoke、stream、batch、async、工具调用和结构化输出接口。

In [6]:
from langchain_deepseek import ChatDeepSeek

deepseek_model = ChatDeepSeek(
    model="deepseek-flash",
    temperature=0,
    timeout=30,
    max_retries=2,
    api_key=api_key,
    base_url=base_url,
)
messages = [
    ("system", "You are a helpful translator. Translate the user sentence to Japanese."),
    ("human", "你好"),
]
print(deepseek_model.invoke(messages).content)

こんにちは


## 3. 使用 ChatOpenAI 兼容接口

DeepSeek 兼容 OpenAI Chat Completions API，因此可以这样调用；正式 LangChain 项目仍优先使用 `ChatDeepSeek`，避免丢失提供商特有字段。

In [7]:
from langchain_openai import ChatOpenAI

compatible_model = ChatOpenAI(
    model="deepseek-flash",
    api_key=api_key,
    base_url=base_url,
    temperature=0,
    timeout=30,
    max_retries=2,
)
print(compatible_model.invoke("Hello").content)

Hello! How can I help you today?


## 3. 使用langchain1.x统一方式
int_chat_model()

In [9]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="deepseek:deepseek-flash", # 最好明确指定供应商
    api_key=api_key,
    base_url=base_url
)
print(model.invoke('介绍下你自己'))

content='你好呀！很高兴认识你！👋\n\n我是 **DeepSeek**，一个由深度求索公司创造的AI助手。让我跟你介绍一下我的“特长”：\n\n## 🌟 我的核心能力\n- **文本理解与生成**：我擅长回答问题、写作、翻译、编程、数学推理等各种任务\n- **超长上下文**：拥有 **1M 上下文**，可以一次性处理像《三体》三部曲那么大体量的内容\n- **文件处理**：支持上传图片、PDF、Word、Excel、PPT、TXT等文件，我会读取其中的文字信息帮你处理\n- **联网搜索**：需要实时信息时，你可以在Web/App上手动开启联网功能\n- **语音输入**：App端支持语音交互\n\n## 💡 我的特点\n- **完全免费**：没有任何收费计划，放心使用！\n- **知识更新**：我的知识截止到2026年2月\n- **多模态能力**：我可以接收你上传的图片，识别和分析其中的可见信息，比如物体、场景、文字等\n\n## 🎯 我能帮你什么？\n无论是学习、工作还是生活中的问题，比如：\n- 写代码、debug\n- 写文章、润色文字\n- 解释复杂概念\n- 头脑风暴、创意策划\n- 数学题、逻辑推理\n- 日常闲聊、倾诉陪伴\n\n我都会尽力帮你！有什么想聊的或者需要帮忙的吗？尽管说～ 😊' additional_kwargs={'refusal': None, 'reasoning_content': '嗯，用户让我介绍一下自己。这是一个非常常见且简单的开场问题。\n\n用户可能刚接触我，想了解我的基本身份、能力和特点，以便后续更好地使用。深层需求可能是想确认我是否适合帮助ta解决特定问题。\n\n我需要给出一个清晰、友好且全面的自我介绍。想到了可以涵盖我的身份（DeepSeek）、核心功能（文本处理、文件支持、联网搜索等）、关键特点（免费、长上下文、知识截止日期）以及我能提供的帮助类型。最后以开放性问题结束，引导对话继续。\n\n回复结构可以这样：先热情打招呼，然后分块介绍核心信息，最后表达乐于帮助的态度并询问具体需求。注意保持信息准确，比如上下文长度、知识截止日期等细节要核对。'} response_metadata={'token_usage': {'completion_tokens': 462, 'prompt_tokens'